## Setup: API Keys and Helper Functions

To use the RapidAPI and SerpAPI, you'll need API keys. Please add them to Colab's secrets manager under the "🔑" icon in the left panel. Name them `RAPIDAPI_KEY` and `SERPAPI_KEY`.

In [50]:
# Used to securely store your API keys
from google.colab import userdata

SERPAPI_KEY = userdata.get('SERPAPI_KEY')

print("API keys loaded from Colab secrets.")

# Define the cookie file name correctly as a string variable
cookie_file = "colab.research.google.com_cookies.txt"

API keys loaded from Colab secrets.


### Install necessary libraries

In [51]:
!pip install requests youtube-transcript-api yt-dlp opencv-python serpapi ffmpeg-python
!pip install --upgrade google-api-python-client # Ensure latest for potential YouTube Data API if needed
!apt-get update && apt-get install -y ffmpeg # Install ffmpeg for video processing

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed

In [52]:
import cv2
import yt_dlp
import os

def extract_multiple_frames_from_url(video_url, timestamps, output_folder='extracted_frames'):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Updated options to handle bot detection
    ydl_opts = {
        'format': 'best[ext=mp4]/best',
        'quiet': True,
        'noplaylist': True,
        'nocheckcertificate': True,
        'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    print(f"Fetching stream URL for: {video_url}")
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(video_url, download=False)
            stream_url = info['url']
    except Exception as e:
        print(f"Failed to extract video info: {e}")
        return

    cap = cv2.VideoCapture(stream_url)
    if not cap.isOpened():
        print("Error: Could not open video stream.")
        return

    for ts in timestamps:
        # Set position in milliseconds
        cap.set(cv2.CAP_PROP_POS_MSEC, ts * 1000)

        success, frame = cap.read()
        if success:
            filename = os.path.join(output_folder, f"frame_at_{ts}s.jpg")
            cv2.imwrite(filename, frame)
            print(f"Successfully saved frame to: {filename}")
        else:
            print(f"Error: Could not read frame at {ts} seconds.")

    cap.release()

target_url = 'https://www.youtube.com/watch?v=xOYsXUsjQII'
time_list = [10, 45.5, 120, 151.8]

extract_multiple_frames_from_url(target_url, time_list)

Fetching stream URL for: https://www.youtube.com/watch?v=xOYsXUsjQII


ERROR: [youtube] xOYsXUsjQII: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


Failed to extract video info: ERROR: [youtube] xOYsXUsjQII: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


In [60]:
import subprocess
import yt_dlp
import os

def extract_multiple_frames_ffmpeg(video_url, timestamps, output_folder='extracted_frames'):
    """
    Uses ffmpeg to extract frames from a YouTube stream URL.
    Note: Requires a 'cookies.txt' file in the current directory if YouTube blocks the request.
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Check if cookies file exists to provide a helpful message
    cookie_file = "colab.research.google.com_cookies.txt"

    ydl_opts = {
        'format': 'best[ext=mp4]/best',
        'quiet': True,
        'noplaylist': True,
        'nocheckcertificate': True,
        'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    if os.path.exists(cookie_file):
        ydl_opts['cookiefile'] = cookie_file
        print(f"Using {cookie_file} for authentication.")
    else:
        print("Warning: 'cookies.txt' not found. If this fails, please upload your exported YouTube cookies.")

    print(f"Fetching stream URL for: {video_url}")
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(video_url, download=False)
            stream_url = info['url']
    except Exception as e:
        print(f"Failed to extract video info: {e}")
        return

    for ts in timestamps:
        output_filename = os.path.join(output_folder, f'frame_{ts}s.jpg')
        command = [
            'ffmpeg', '-y', '-ss', str(ts), '-i', stream_url,
            '-vframes', '1', '-q:v', '2', output_filename
        ]
        try:
            subprocess.run(command, check=True, capture_output=True)
            print(f"  Successfully saved: {output_filename}")
        except subprocess.CalledProcessError as e:
            print(f"  Error at {ts}s: {e.stderr.decode()}")

youtube_url = 'https://youtu.be/Ryse7WB0yfw?si=yZpRYSW6kX7OtD1u'
time_list = [10, 45.5, 120, 151.8]

extract_multiple_frames_ffmpeg(youtube_url, time_list)

Using colab.research.google.com_cookies.txt for authentication.
Fetching stream URL for: https://youtu.be/Ryse7WB0yfw?si=yZpRYSW6kX7OtD1u


ERROR: [youtube] Ryse7WB0yfw: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


Failed to extract video info: ERROR: [youtube] Ryse7WB0yfw: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


In [54]:
import subprocess
import yt_dlp
import os
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List
import nest_asyncio
from threading import Thread

app = FastAPI(title="YouTube Frame Extractor API")

class ExtractionRequest(BaseModel):
    url: str
    timestamps: List[float]
    output_folder: str = 'extracted_frames'

def get_stream_url(video_url):
    cookie_file = "colab.research.google.com_cookies.txt"
    ydl_opts = {
        'format': 'best[ext=mp4]/best',
        'quiet': True,
        'noplaylist': True,
        'nocheckcertificate': True,
        'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    # Use the uploaded cookie file if it exists
    if os.path.exists(cookie_file):
        ydl_opts['cookiefile'] = cookie_file

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(video_url, download=False)
        return info['url']

@app.post("/extract")
async def extract_frames(request: ExtractionRequest):
    if not os.path.exists(request.output_folder):
        os.makedirs(request.output_folder)

    try:
        stream_url = get_stream_url(request.url)
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Could not fetch stream: {str(e)}")

    saved_files = []
    for ts in request.timestamps:
        output_filename = os.path.join(request.output_folder, f'frame_{ts}s.jpg')
        command = [
            'ffmpeg', '-y', '-ss', str(ts), '-i', stream_url,
            '-vframes', '1', '-q:v', '2', output_filename
        ]
        result = subprocess.run(command, capture_output=True)
        if result.returncode == 0:
            saved_files.append(output_filename)

    return {"status": "success", "extracted_frames": saved_files}

def run_app():
    nest_asyncio.apply()
    # Use a different port if 8000 is still bound, or just restart
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Restart server logic: normally you'd stop the old thread,
# but in Colab rerunning the cell starts a new one or you can restart the runtime.
thread = Thread(target=run_app)
thread.start()

print("FastAPI server updated with cookie support.")
print("You can now send POST requests to the public ngrok URL.")

FastAPI server updated with cookie support.
You can now send POST requests to the public ngrok URL.


INFO:     Started server process [4630]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
